In [143]:
import torch

torch.set_printoptions(precision=2, sci_mode=False)

In [144]:
inputs = torch.randn(2, 6, 3)
inputs

tensor([[[-0.04, -0.99, -0.66],
         [-0.35,  1.45, -0.62],
         [-0.60,  2.39,  0.18],
         [-1.60, -2.02, -0.40],
         [-0.06,  0.70, -0.90],
         [ 0.84, -1.39, -0.63]],

        [[ 1.70, -1.32, -0.20],
         [-0.25, -0.32,  1.54],
         [-0.99,  0.20, -0.68],
         [ 0.20, -0.19, -1.00],
         [ 1.20, -1.19,  0.15],
         [ 0.23,  0.36, -0.98]]])

In [145]:
class CasAtten(torch.nn.Module):
    def __init__(
        self,
        d_in,
        d_out,
        context_len,
        qkv_bias=False,
        drop_out_p=0.2,
    ):
        super().__init__()
        self.W_Q = torch.nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_K = torch.nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_V = torch.nn.Linear(d_in, d_out, bias=qkv_bias)
        self.drop_out = torch.nn.Dropout(drop_out_p)
        self.register_buffer(
            "mask", torch.triu(torch.ones(context_len, context_len), diagonal=1)
        )

    def forward(self, x):
        q = self.W_Q(x)
        k = self.W_K(x)
        v = self.W_V(x)
        atten = q @ k.transpose(1, 2)
        # atten.masked_fill_(self.mask.bool(), -torch.inf)
        # in inference this shape can be changed but in training it stays same mostly
        atten.masked_fill_(self.mask[: x.shape[1], : x.shape[1]].bool(), -torch.inf)
        atten = self.drop_out(torch.softmax(atten / k.shape[-1] ** 0.5, dim=-1))
        return atten @ v

In [146]:
class MultiHeadAttention(torch.nn.Module):
    def __init__(self, d_in, d_out, context_len, num_heads=3):
        super().__init__()
        self.heads = torch.nn.ModuleList(
            [CasAtten(d_in, d_out, context_len) for _ in range(num_heads)]
        )
        # self.heads = torch.nn.Sequential(
        #     CasAtten(d_in, d_out, context_len),
        #     CasAtten(d_in, d_out, context_len),
        #     CasAtten(d_in, d_out, context_len),
        # )

    def forward(self, x):
        return torch.concat([head(x) for head in self.heads], dim=-1)
        # return self.heads(x)

In [147]:
torch.manual_seed(31)
mha = MultiHeadAttention(3, 2, 6, num_heads=3)
# mha
mha(inputs)

tensor([[[ 0.61,  0.64, -0.56,  0.75, -0.01,  0.52],
         [-0.66, -0.13,  0.50, -0.54, -0.55, -0.09],
         [-1.17, -0.54,  0.83, -1.09, -0.64, -0.29],
         [ 0.49,  0.27,  0.41, -0.87, -0.38, -0.29],
         [-0.47, -0.05,  0.14, -0.43, -0.62, -0.08],
         [ 0.21,  0.32,  0.05, -0.06, -0.63, -0.48]],

        [[ 0.99,  0.91, -0.30,  1.90,  1.01,  0.41],
         [ 0.66,  0.02, -0.10,  0.63,  1.09,  0.42],
         [ 0.39,  0.13, -0.02, -0.28,  0.46,  0.29],
         [ 0.29,  0.24, -0.18,  0.28,  0.11,  0.18],
         [ 0.41,  0.26, -0.26,  0.69,  0.47,  0.19],
         [ 0.28,  0.32, -0.07,  0.64,  0.22,  0.11]]], grad_fn=<CatBackward0>)

In [148]:
# Single multi head

In [149]:
torch.manual_seed(31)
inputs = torch.randn(1, 5, 3)
inputs

tensor([[[ 0.35, -0.57, -0.39],
         [-0.29, -0.45,  0.92],
         [-0.76,  1.06, -0.12],
         [ 0.58,  0.47,  0.30],
         [ 0.56,  2.29,  2.11]]])

In [150]:
d_in = 3  # embedding_size
d_out = 4  # projection

In [151]:
num_heads = 2
head_dim = d_out // num_heads
head_dim

2

In [152]:
torch.manual_seed(31)
W_Q = torch.nn.Linear(d_in, d_out, bias=False)
W_K = torch.nn.Linear(d_in, d_out, bias=False)
W_V = torch.nn.Linear(d_in, d_out, bias=False)
W_Q.weight.shape, W_K.weight.shape, W_V.weight.shape

(torch.Size([4, 3]), torch.Size([4, 3]), torch.Size([4, 3]))

In [153]:
queries, keys, values = W_Q(inputs), W_K(inputs), W_V(inputs)
queries.shape, queries

(torch.Size([1, 5, 4]),
 tensor([[[ 0.04,  0.19,  0.16,  0.26],
          [-0.11,  0.18, -0.26,  0.19],
          [ 0.17, -0.38, -0.11, -0.36],
          [-0.30, -0.14,  0.01, -0.41],
          [-0.93, -0.72, -0.46, -1.58]]], grad_fn=<UnsafeViewBackward0>))

In [154]:
k_h = keys.view(1, 5, num_heads, head_dim)
v_h = values.view(1, 5, num_heads, head_dim)
q_h = queries.view(1, 5, num_heads, head_dim)
q_h.shape

torch.Size([1, 5, 2, 2])

In [155]:
k_h = k_h.transpose(1, 2)
v_h = v_h.transpose(1, 2)
q_h = q_h.transpose(1, 2)
q_h.shape, queries

(torch.Size([1, 2, 5, 2]),
 tensor([[[ 0.04,  0.19,  0.16,  0.26],
          [-0.11,  0.18, -0.26,  0.19],
          [ 0.17, -0.38, -0.11, -0.36],
          [-0.30, -0.14,  0.01, -0.41],
          [-0.93, -0.72, -0.46, -1.58]]], grad_fn=<UnsafeViewBackward0>))

In [156]:
k_h, k_h.transpose(2, 3), k_h.transpose(2, 3).shape

(tensor([[[[ 0.30,  0.37],
           [ 0.31, -0.32],
           [-0.63, -0.36],
           [-0.20, -0.13],
           [-1.05, -1.34]],
 
          [[-0.46, -0.34],
           [ 0.41,  0.13],
           [ 0.52,  0.22],
           [ 0.01,  0.50],
           [ 1.29,  2.14]]]], grad_fn=<TransposeBackward0>),
 tensor([[[[ 0.30,  0.31, -0.63, -0.20, -1.05],
           [ 0.37, -0.32, -0.36, -0.13, -1.34]],
 
          [[-0.46,  0.41,  0.52,  0.01,  1.29],
           [-0.34,  0.13,  0.22,  0.50,  2.14]]]], grad_fn=<TransposeBackward0>),
 torch.Size([1, 2, 2, 5]))

In [157]:
q_h.shape, k_h.transpose(2, 3).shape

(torch.Size([1, 2, 5, 2]), torch.Size([1, 2, 2, 5]))

In [158]:
atten_scores = q_h @ k_h.transpose(2, 3)
atten_scores.shape, atten_scores

(torch.Size([1, 2, 5, 5]),
 tensor([[[[     0.08,     -0.05,     -0.09,     -0.03,     -0.30],
           [     0.03,     -0.09,      0.00,     -0.00,     -0.13],
           [    -0.09,      0.18,      0.03,      0.02,      0.34],
           [    -0.14,     -0.05,      0.24,      0.08,      0.50],
           [    -0.54,     -0.06,      0.84,      0.28,      1.94]],
 
          [[    -0.16,      0.10,      0.14,      0.13,      0.77],
           [     0.06,     -0.08,     -0.09,      0.09,      0.07],
           [     0.17,     -0.09,     -0.14,     -0.18,     -0.91],
           [     0.13,     -0.05,     -0.08,     -0.21,     -0.86],
           [     0.75,     -0.40,     -0.59,     -0.80,     -3.98]]]],
        grad_fn=<UnsafeViewBackward0>))

In [159]:
atten_scores.masked_fill_(
    torch.triu(
        torch.ones(5, 5),
        diagonal=1,
    ).bool(),
    -torch.inf,
)

tensor([[[[ 0.08,  -inf,  -inf,  -inf,  -inf],
          [ 0.03, -0.09,  -inf,  -inf,  -inf],
          [-0.09,  0.18,  0.03,  -inf,  -inf],
          [-0.14, -0.05,  0.24,  0.08,  -inf],
          [-0.54, -0.06,  0.84,  0.28,  1.94]],

         [[-0.16,  -inf,  -inf,  -inf,  -inf],
          [ 0.06, -0.08,  -inf,  -inf,  -inf],
          [ 0.17, -0.09, -0.14,  -inf,  -inf],
          [ 0.13, -0.05, -0.08, -0.21,  -inf],
          [ 0.75, -0.40, -0.59, -0.80, -3.98]]]],
       grad_fn=<MaskedFillBackward0>)

In [160]:
atten_norm = torch.softmax(atten_scores / keys.shape[-1] ** 0.5, dim=-1)
atten_norm.shape, atten_norm

(torch.Size([1, 2, 5, 5]),
 tensor([[[[1.00, 0.00, 0.00, 0.00, 0.00],
           [0.52, 0.48, 0.00, 0.00, 0.00],
           [0.31, 0.36, 0.33, 0.00, 0.00],
           [0.23, 0.24, 0.28, 0.26, 0.00],
           [0.11, 0.14, 0.22, 0.16, 0.37]],
 
          [[1.00, 0.00, 0.00, 0.00, 0.00],
           [0.52, 0.48, 0.00, 0.00, 0.00],
           [0.37, 0.32, 0.31, 0.00, 0.00],
           [0.27, 0.25, 0.25, 0.23, 0.00],
           [0.38, 0.21, 0.19, 0.18, 0.04]]]], grad_fn=<SoftmaxBackward0>))

In [161]:
atten_norm_d = torch.nn.Dropout(0.2)(atten_norm)
atten_norm_d.shape, atten_norm_d

(torch.Size([1, 2, 5, 5]),
 tensor([[[[1.25, 0.00, 0.00, 0.00, 0.00],
           [0.64, 0.61, 0.00, 0.00, 0.00],
           [0.39, 0.45, 0.41, 0.00, 0.00],
           [0.00, 0.30, 0.35, 0.00, 0.00],
           [0.00, 0.00, 0.27, 0.20, 0.00]],
 
          [[1.25, 0.00, 0.00, 0.00, 0.00],
           [0.65, 0.60, 0.00, 0.00, 0.00],
           [0.46, 0.40, 0.39, 0.00, 0.00],
           [0.34, 0.31, 0.31, 0.29, 0.00],
           [0.47, 0.00, 0.24, 0.22, 0.04]]]], grad_fn=<MulBackward0>))

In [163]:
(atten_norm_d @ v_h).shape, (atten_norm_d @ v_h)

(torch.Size([1, 2, 5, 2]),
 tensor([[[[-0.50,  0.19],
           [ 0.01,  0.33],
           [ 0.21, -0.05],
           [ 0.28, -0.12],
           [ 0.11, -0.13]],
 
          [[-0.19,  0.64],
           [-0.40,  0.35],
           [-0.11, -0.11],
           [ 0.02, -0.10],
           [ 0.15, -0.04]]]], grad_fn=<UnsafeViewBackward0>))

In [164]:
context_vec = (atten_norm_d @ v_h).transpose(1, 2)
context_vec.shape, context_vec

(torch.Size([1, 5, 2, 2]),
 tensor([[[[-0.50,  0.19],
           [-0.19,  0.64]],
 
          [[ 0.01,  0.33],
           [-0.40,  0.35]],
 
          [[ 0.21, -0.05],
           [-0.11, -0.11]],
 
          [[ 0.28, -0.12],
           [ 0.02, -0.10]],
 
          [[ 0.11, -0.13],
           [ 0.15, -0.04]]]], grad_fn=<TransposeBackward0>))

In [ ]:
# contiguous is needed here to bring all the tensor elements in the continous memory
# physical GPU memory otherwise view is not gonna work
context_vec.contiguous().view(1, 5, d_out)

tensor([[[-0.50,  0.19, -0.19,  0.64],
         [ 0.01,  0.33, -0.40,  0.35],
         [ 0.21, -0.05, -0.11, -0.11],
         [ 0.28, -0.12,  0.02, -0.10],
         [ 0.11, -0.13,  0.15, -0.04]]], grad_fn=<ViewBackward0>)

In [220]:
torch.manual_seed(31)


class MHAttention(torch.nn.Module):
    def __init__(
        self,
        d_in,  # also embedding size
        d_out,
        num_heads,
        context_len,  # num_of_tokens in training but could be different in inferences
        dropout_p=0.2,
        qkv_bias=False,
    ):
        super().__init__()
        self.num_heads = num_heads
        self.d_out = d_out
        assert d_out % num_heads == 0
        self.head_dim = d_out // num_heads
        self.WQ = torch.nn.Linear(d_in, d_out, bias=qkv_bias)
        self.WK = torch.nn.Linear(d_in, d_out, bias=qkv_bias)
        self.WV = torch.nn.Linear(d_in, d_out, bias=qkv_bias)
        self.register_buffer(
            "mask", torch.triu(torch.ones(context_len, context_len), diagonal=1)
        )
        self.DO = torch.nn.Dropout(dropout_p)
        # project layer (optional)
        self.PO = torch.nn.Linear(d_out, d_out)

    def forward(self, inputs):
        b, num_tokens, emb_size = inputs.shape
        q_vec = self.WQ(inputs)
        k_vec = self.WK(inputs)
        v_vec = self.WV(inputs)
        print(q_vec.shape)
        # let's divide them to have head dims
        q_vec = q_vec.view(b, num_tokens, self.num_heads, self.head_dim)
        k_vec = k_vec.view(b, num_tokens, self.num_heads, self.head_dim)
        v_vec = v_vec.view(b, num_tokens, self.num_heads, self.head_dim)
        print(q_vec.shape)
        # bring head dim on the batching side inside of mat mul side
        q_vec = q_vec.transpose(1, 2)
        k_vec = k_vec.transpose(1, 2)
        v_vec = v_vec.transpose(1, 2)
        print(q_vec.shape)
        # now let's start the drop_out(softmax(mask(Q.KT) / d_in ** 0.5)) @ V
        # as the heads are in batch side of matrix, they will be treated as separate things
        # mat mul will not use them
        atten_scores = q_vec @ k_vec.transpose(2, 3)
        # atten_scores.masked_fill_(self.mask.bool(), -torch.inf)
        atten_scores.masked_fill_(
            self.mask.bool()[:num_tokens, :num_tokens], -torch.inf
        )
        atten_scores = torch.softmax(atten_scores / k_vec.shape[-1] ** 0.5, dim=-1)
        atten_scores = self.DO(atten_scores)
        context_vec = atten_scores @ v_vec
        # bring head-dim back to its location
        context_vec = context_vec.transpose(2, 1)
        # make it a single output vector now
        context_vec = context_vec.contiguous().view(b, num_tokens, self.d_out)
        print(context_vec.shape)
        context_vec = self.PO(context_vec)
        print(context_vec.shape)
        return context_vec  # , k_vec


# 1 (batch), 2 (num_of_tokens/num_of_words -> I am), 2 (embedding size)
batch, num_tokens, emb_size, d_out = 1, 2, 2, 6
num_heads = 3
inputs = torch.randn(batch, num_tokens, emb_size)
print(inputs.shape, inputs)
print("=====")
mha = MHAttention(
    d_in=emb_size,
    d_out=d_out,
    num_heads=num_heads,
    context_len=num_tokens,
)
g = mha(inputs)
g

torch.Size([1, 2, 2]) tensor([[[ 0.35, -0.57],
         [-0.39, -0.29]]])
=====
torch.Size([1, 2, 6])
torch.Size([1, 2, 3, 2])
torch.Size([1, 3, 2, 2])
torch.Size([1, 2, 6])
torch.Size([1, 2, 6])


tensor([[[ 0.29,  0.29, -0.05,  0.28, -0.01, -0.12],
         [ 0.22,  0.26, -0.17,  0.23,  0.20, -0.20]]], grad_fn=<ViewBackward0>)

In [219]:
from torchviz import make_dot


In [221]:
graph = make_dot(g, params=dict(mha.named_parameters()))

graph.render("model_graph", format="png")

'model_graph.png'

In [224]:
torch.manual_seed(31)
ips = torch.randn(5, 5, 768)
gpt_2 = MHAttention(
    d_in=768,
    d_out=768,
    context_len=1024,
    num_heads=12,
)
gpt_2(ips).shape

torch.Size([5, 5, 768])
torch.Size([5, 5, 12, 64])
torch.Size([5, 12, 5, 64])
torch.Size([5, 5, 768])
torch.Size([5, 5, 768])


torch.Size([5, 5, 768])

In [ ]:
sum([param.numel() for param in gpt_2.parameters() if param.requires_grad == True])

2360064